# Traffic Demand Prediction (Time-Aware)

This notebook builds a robust regression pipeline for traffic demand prediction and avoids temporal leakage.

Why this version:
- The test set represents a future time window.
- Random validation overestimates performance.
- We use **time-aware validation** (`day=48` train, `day=49` validation) to better match leaderboard behavior.

Models compared:
- Random Forest Regressor
- Gradient Boosting Regressor
- XGBoost Regressor
- LightGBM Regressor

Evaluation metric: **R2 score**.


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

RANDOM_STATE = 42
sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 120)


In [ ]:
# Load data
train_df = pd.read_csv('dataset/train.csv')
test_df = pd.read_csv('dataset/test.csv')
sample_sub = pd.read_csv('dataset/sample_submission.csv')

print('Train shape:', train_df.shape)
print('Test shape :', test_df.shape)
print('Sample submission shape:', sample_sub.shape)

display(train_df.head())
display(test_df.head())


In [ ]:
# Data quality checks
print('Train missing %:')
display((train_df.isna().mean() * 100).sort_values(ascending=False).to_frame('missing_pct'))

print('Test missing %:')
display((test_df.isna().mean() * 100).sort_values(ascending=False).to_frame('missing_pct'))

print('Unique values (categorical columns):')
for c in ['RoadType', 'Weather', 'LargeVehicles', 'Landmarks']:
    vals = train_df[c].fillna('Unknown').unique().tolist()
    print(f'- {c}: {vals}')


In [ ]:
# Temporal coverage check (important)
def add_time_cols(df):
    hm = df['timestamp'].str.split(':', expand=True).astype(int)
    out = df.copy()
    out['hour'] = hm[0]
    out['minute'] = hm[1]
    out['slot'] = out['hour'] * 4 + out['minute'] // 15
    return out

train_tmp = add_time_cols(train_df)
test_tmp = add_time_cols(test_df)

print('Train day counts:', train_tmp['day'].value_counts().sort_index().to_dict())
print('Test day counts :', test_tmp['day'].value_counts().sort_index().to_dict())
print('Train slot min/max:', train_tmp['slot'].min(), train_tmp['slot'].max())
print('Test slot min/max :', test_tmp['slot'].min(), test_tmp['slot'].max())

overlap = len(set(zip(train_tmp['day'], train_tmp['slot'])) & set(zip(test_tmp['day'], test_tmp['slot'])))
print('Train/Test (day, slot) overlap:', overlap)


## Feature Engineering + Preprocessing

Design goals:
- use temporal, road, weather, and spatial signals
- avoid leakage-prone target encoding for validation/test
- keep encodings stable for unseen categories


In [ ]:
def add_base_features(df: pd.DataFrame) -> pd.DataFrame:
    data = df.copy()
    hm = data['timestamp'].str.split(':', expand=True).astype(int)
    data['hour'] = hm[0]
    data['minute'] = hm[1]
    data['slot'] = data['hour'] * 4 + (data['minute'] // 15)

    data['hour_sin'] = np.sin(2 * np.pi * data['hour'] / 24)
    data['hour_cos'] = np.cos(2 * np.pi * data['hour'] / 24)
    data['slot_sin'] = np.sin(2 * np.pi * data['slot'] / 96)
    data['slot_cos'] = np.cos(2 * np.pi * data['slot'] / 96)

    data['RoadType'] = data['RoadType'].fillna('Unknown')
    data['Weather'] = data['Weather'].fillna('Unknown')

    data['LargeVehicles_bin'] = (data['LargeVehicles'] == 'Allowed').astype(int)
    data['Landmarks_bin'] = (data['Landmarks'] == 'Yes').astype(int)
    data['temp_missing'] = data['Temperature'].isna().astype(int)

    data['geo3'] = data['geohash'].str[:3]
    data['geo4'] = data['geohash'].str[:4]
    return data


class RobustPreprocessor:
    def __init__(self):
        self.weather_temp_median = None
        self.global_temp_median = None
        self.freq_maps = {}
        self.enc_maps = {}
        self.features = [
            'day', 'hour', 'minute', 'slot',
            'hour_sin', 'hour_cos', 'slot_sin', 'slot_cos',
            'NumberofLanes', 'LargeVehicles_bin', 'Landmarks_bin',
            'Temperature', 'temp_missing',
            'temp_x_lanes', 'is_highway', 'lane_x_landmark', 'lane_x_largevehicle',
            'geohash_enc', 'geo3_enc', 'geo4_enc', 'RoadType_enc', 'Weather_enc',
            'geohash_freq', 'geo3_freq', 'geo4_freq', 'RoadType_freq', 'Weather_freq',
        ]

    def fit(self, df: pd.DataFrame):
        d = add_base_features(df)
        self.weather_temp_median = d.groupby('Weather')['Temperature'].median().to_dict()
        self.global_temp_median = float(d['Temperature'].median())

        for c in ['geohash', 'geo3', 'geo4', 'RoadType', 'Weather']:
            self.freq_maps[c] = d[c].value_counts(normalize=True).to_dict()
            vals = sorted(d[c].unique())
            self.enc_maps[c] = {v: i for i, v in enumerate(vals)}
        return self

    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        d = add_base_features(df)

        d['Temperature'] = d['Temperature'].fillna(d['Weather'].map(self.weather_temp_median)).fillna(self.global_temp_median)

        for c in ['geohash', 'geo3', 'geo4', 'RoadType', 'Weather']:
            d[f'{c}_freq'] = d[c].map(self.freq_maps[c]).fillna(0.0)
            d[f'{c}_enc'] = d[c].map(self.enc_maps[c]).fillna(-1).astype(int)

        d['temp_x_lanes'] = d['Temperature'] * d['NumberofLanes']
        d['is_highway'] = (d['RoadType'] == 'Highway').astype(int)
        d['lane_x_landmark'] = d['NumberofLanes'] * d['Landmarks_bin']
        d['lane_x_largevehicle'] = d['NumberofLanes'] * d['LargeVehicles_bin']

        X = d[self.features].copy()
        X = X.replace([np.inf, -np.inf], np.nan).fillna(0)
        return X


In [ ]:
# Time-aware split for validation
train_part = train_df[train_df['day'] == 48].copy()
valid_part = train_df[train_df['day'] == 49].copy()

print('Train rows (day=48):', train_part.shape[0])
print('Valid rows (day=49):', valid_part.shape[0])

pre = RobustPreprocessor().fit(train_part)
X_train = pre.transform(train_part)
X_valid = pre.transform(valid_part)

y_train = train_part['demand'].values
y_valid = valid_part['demand'].values

print('Feature count:', X_train.shape[1])


In [ ]:
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))


def build_models(seed=42):
    return {
        'RandomForest': RandomForestRegressor(
            n_estimators=500,
            min_samples_leaf=2,
            max_features='sqrt',
            random_state=seed,
            n_jobs=-1,
        ),
        'GradientBoosting': GradientBoostingRegressor(
            n_estimators=700,
            learning_rate=0.03,
            max_depth=4,
            subsample=0.9,
            random_state=seed,
        ),
        'XGBoost': XGBRegressor(
            n_estimators=900,
            learning_rate=0.04,
            max_depth=8,
            subsample=0.9,
            colsample_bytree=0.9,
            objective='reg:squarederror',
            random_state=seed,
            n_jobs=-1,
        ),
        'LightGBM': LGBMRegressor(
            n_estimators=1200,
            learning_rate=0.03,
            num_leaves=63,
            subsample=0.9,
            colsample_bytree=0.9,
            random_state=seed,
            n_jobs=-1,
        ),
    }


results = []
models = build_models(RANDOM_STATE)
fitted_models = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    pred = np.clip(model.predict(X_valid), 0.0, 1.0)

    row = {
        'Model': name,
        'R2': r2_score(y_valid, pred),
        'MAE': mean_absolute_error(y_valid, pred),
        'RMSE': rmse(y_valid, pred),
    }
    results.append(row)
    fitted_models[name] = model
    print(f"{name:<18} | R2={row['R2']:.6f} | MAE={row['MAE']:.6f} | RMSE={row['RMSE']:.6f}")

results_df = pd.DataFrame(results).sort_values('R2', ascending=False).reset_index(drop=True)
display(results_df)

best_model_name = results_df.loc[0, 'Model']
print('Best model (time-aware validation):', best_model_name)


In [ ]:
# Model comparison chart
plt.figure(figsize=(9, 4))
sns.barplot(data=results_df, x='Model', y='R2', palette='viridis')
plt.title('Validation R2 by Model (Time-Aware)')
plt.ylim(results_df['R2'].min() - 0.02, results_df['R2'].max() + 0.02)
plt.show()


In [ ]:
# Retrain best model on full train and predict test
pre_full = RobustPreprocessor().fit(train_df)
X_full = pre_full.transform(train_df)
X_test = pre_full.transform(test_df)
y_full = train_df['demand'].values

final_model = build_models(RANDOM_STATE)[best_model_name]
final_model.fit(X_full, y_full)

test_pred = np.clip(final_model.predict(X_test), 0.0, 1.0)

submission = pd.DataFrame({
    'Index': test_df['Index'],
    'demand': test_pred,
})

submission.to_csv('submission_best_model.csv', index=False)
print('Saved: submission_best_model.csv')
print('Submission shape:', submission.shape)
display(submission.head())


In [ ]:
# Final checks
assert submission.shape == (41778, 2)
assert submission.columns.tolist() == ['Index', 'demand']
assert submission['Index'].equals(test_df['Index'])
assert np.issubdtype(submission['demand'].dtype, np.number)
assert submission['demand'].isna().sum() == 0

print('All submission checks passed.')
